In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("cardekho_dataset.csv")

df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [3]:
df.shape

(15411, 14)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  str    
 2   brand              15411 non-null  str    
 3   model              15411 non-null  str    
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  str    
 7   fuel_type          15411 non-null  str    
 8   transmission_type  15411 non-null  str    
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), str(6)
memory usage: 1.6 MB


In [5]:
df.columns.tolist()

['Unnamed: 0',
 'car_name',
 'brand',
 'model',
 'vehicle_age',
 'km_driven',
 'seller_type',
 'fuel_type',
 'transmission_type',
 'mileage',
 'engine',
 'max_power',
 'seats',
 'selling_price']

In [6]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,15411.0,NaN,NaN,NaN,9811.857699,5643.418542,0.0,4906.5,9872.0,14668.5,19543.0
car_name,15411,121,Hyundai i20,906,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand,15411,32,Maruti,4992,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model,15411,120,i20,906,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vehicle_age,15411.0,NaN,NaN,NaN,6.036338,3.013291,0.0,4.0,6.0,8.0,29.0
km_driven,15411.0,NaN,NaN,NaN,55616.480631,51618.548422,100.0,30000.0,50000.0,70000.0,3800000.0
seller_type,15411,3,Dealer,9539,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fuel_type,15411,5,Petrol,7643,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transmission_type,15411,2,Manual,12225,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mileage,15411.0,NaN,NaN,NaN,19.701151,4.171265,4.0,17.0,19.67,22.7,33.54


In [7]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [8]:
df = df.drop(columns=["Unnamed: 0"])

In [9]:
df.shape

(15411, 13)

In [10]:
df.duplicated().sum()

np.int64(167)

In [11]:
df[df["seats"] == 0]

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
3217,Honda City,Honda,City,18,40000,Individual,Petrol,Manual,13.00,1493,100.00,0,115000
12619,Nissan Kicks,Nissan,Kicks,2,10000,Individual,Diesel,Manual,19.39,1461,108.49,0,1154000


In [12]:
(df["seats"] == 0).sum()

np.int64(2)

In [13]:
df.sort_values("km_driven", ascending=False)[
    ["car_name", "brand", "vehicle_age", "km_driven", "selling_price"]
].head(15)

,car_name,brand,vehicle_age,km_driven,selling_price
15409,Mahindra XUV500,Mahindra,5,3800000,1225000
11903,Mercedes-Benz E-Class,Mercedes-Benz,9,1325000,1350000
8772,Toyota Innova,Toyota,6,950000,950000
4532,Renault Duster,Renault,4,850000,850000
6712,Volkswagen Vento,Volkswagen,9,830000,380000
10838,Hyundai Creta,Hyundai,6,825000,825000
11615,Volkswagen Polo,Volkswagen,5,820000,695000
7323,Ford Ecosport,Ford,6,720000,675000
9170,Skoda Rapid,Skoda,5,675000,675000
7534,Toyota Fortuner,Toyota,2,590000,2850000


In [14]:
df = df.drop_duplicates().reset_index(drop=True)

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
df[df["seats"] == 0]

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
3207,Honda City,Honda,City,18,40000,Individual,Petrol,Manual,13.00,1493,100.00,0,115000
12504,Nissan Kicks,Nissan,Kicks,2,10000,Individual,Diesel,Manual,19.39,1461,108.49,0,1154000


In [17]:
Q1 = df["km_driven"].quantile(0.25)
Q3 = df["km_driven"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

lower_bound, upper_bound

(np.float64(-30000.0), np.float64(130000.0))

In [18]:
(df["km_driven"] > upper_bound).sum()

np.int64(464)

In [19]:
df[df["km_driven"] > upper_bound][
    ["car_name", "vehicle_age", "km_driven", "selling_price"]
].sort_values("km_driven", ascending=False)

,car_name,vehicle_age,km_driven,selling_price
15242,Mahindra XUV500,5,3800000,1225000
11790,Mercedes-Benz E-Class,9,1325000,1350000
8701,Toyota Innova,6,950000,950000
4516,Renault Duster,4,850000,850000
6673,Volkswagen Vento,9,830000,380000
...,...,...,...,...
8052,Maruti Wagon R,10,130890,160000
8567,Hyundai Elantra,8,130600,525000
10966,Mahindra Scorpio,10,130311,385000
5337,Skoda Rapid,4,130200,650000


In [20]:
(df["km_driven"] > 300000).sum()

np.int64(30)

In [21]:
df[df["km_driven"] > 300000][
    ["car_name", "brand", "vehicle_age", "km_driven", "selling_price"]
].sort_values("km_driven", ascending=False)

,car_name,brand,vehicle_age,km_driven,selling_price
15242,Mahindra XUV500,Mahindra,5,3800000,1225000
11790,Mercedes-Benz E-Class,Mercedes-Benz,9,1325000,1350000
8701,Toyota Innova,Toyota,6,950000,950000
4516,Renault Duster,Renault,4,850000,850000
6673,Volkswagen Vento,Volkswagen,9,830000,380000
10742,Hyundai Creta,Hyundai,6,825000,825000
11508,Volkswagen Polo,Volkswagen,5,820000,695000
7276,Ford Ecosport,Ford,6,720000,675000
9089,Skoda Rapid,Skoda,5,675000,675000
7486,Toyota Fortuner,Toyota,2,590000,2850000


In [22]:
df = df[df["km_driven"] <= 300000].copy()

In [23]:
df["km_driven"].describe()

count     15214.000000
mean      54457.378664
std       34579.213193
min         100.000000
25%       30000.000000
50%       50000.000000
75%       70000.000000
max      300000.000000
Name: km_driven, dtype: float64

In [24]:
df[df["seats"] == 0][
    ["car_name", "brand", "model", "seats"]
]

,car_name,brand,model,seats
3207,Honda City,Honda,City,0
12504,Nissan Kicks,Nissan,Kicks,0


In [25]:
df.loc[df["car_name"] == "Honda City", "seats"] = 5
df.loc[df["car_name"] == "Nissan Kicks", "seats"] = 5

In [26]:
df[df["seats"] == 0]


,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price


In [27]:
df.shape

(15214, 13)

In [28]:
df[[
    "vehicle_age",
    "km_driven",
    "mileage",
    "engine",
    "max_power",
    "seats",
    "selling_price"
]].describe().T

,count,mean,std,min,25%,50%,75%,max
vehicle_age,15214.0,6.037531,3.014498,0.0,4.0,6.00,8.0,29.00
km_driven,15214.0,54457.378664,34579.213193,100.0,30000.0,50000.00,70000.0,300000.00
mileage,15214.0,19.701267,4.167436,4.0,17.0,19.67,22.7,33.54
engine,15214.0,1485.468844,519.935380,793.0,1197.0,1248.00,1582.0,6592.00
max_power,15214.0,100.617170,42.933491,38.4,74.0,88.50,117.3,626.00
seats,15214.0,5.325687,0.805278,2.0,5.0,5.00,5.0,9.00
selling_price,15214.0,774995.587945,895249.656682,40000.0,385000.0,559500.00,825000.0,39500000.00


In [29]:
df["price_lakh"] = df["selling_price"] / 100000

df["km_per_year"] = np.where(
    df["vehicle_age"] > 0,
    df["km_driven"] / df["vehicle_age"],
    df["km_driven"]
)

df["age_group"] = pd.cut(
    df["vehicle_age"],
    bins=[-1, 2, 5, 8, 12, 30],
    labels=["0-2 Years", "3-5 Years", "6-8 Years", "9-12 Years", "13+ Years"]
)

df["km_band"] = pd.cut(
    df["km_driven"],
    bins=[0, 25000, 50000, 75000, 100000, 150000, 300000],
    labels=[
        "0-25k",
        "25k-50k",
        "50k-75k",
        "75k-100k",
        "100k-150k",
        "150k+"
    ]
)

In [30]:
df["power_per_litre"] = df["max_power"] / (df["engine"] / 1000)

In [31]:
df[[
    "car_name",
    "vehicle_age",
    "km_driven",
    "price_lakh",
    "km_per_year",
    "age_group",
    "km_band",
    "power_per_litre"
]].head()

,car_name,vehicle_age,km_driven,price_lakh,km_per_year,age_group,km_band,power_per_litre
0,Maruti Alto,9,120000,1.20,13333.333333,9-12 Years,100k-150k,58.165829
1,Hyundai Grand,5,20000,5.50,4000.000000,3-5 Years,0-25k,68.504595
2,Hyundai i20,11,60000,2.15,5454.545455,9-12 Years,50k-75k,66.833751
3,Maruti Alto,9,37000,2.26,4111.111111,9-12 Years,25k-50k,67.234469
4,Ford Ecosport,6,30000,5.70,5000.000000,6-8 Years,25k-50k,65.814419


In [32]:
df[[
    "price_lakh",
    "km_per_year",
    "age_group",
    "km_band",
    "power_per_litre"
]].isnull().sum()

price_lakh         0
km_per_year        0
age_group          0
km_band            0
power_per_litre    0
dtype: int64

In [34]:
price_99 = df["price_lakh"].quantile(0.99)

print("99th percentile:", round(price_99, 2), "Lakh")

99th percentile: 45.43 Lakh


In [35]:
fig = px.histogram(
    df[df["price_lakh"] <= price_99],
    x="price_lakh",
    nbins=50,
    title="Distribution of Used Car Selling Prices (Up to 99th Percentile)",
    labels={"price_lakh": "Selling Price (₹ Lakh)"}
)

fig.update_layout(
    xaxis_title="Selling Price (₹ Lakh)",
    yaxis_title="Number of Cars"
)

fig.show()

In [36]:
fig = px.box(
    df,
    x="price_lakh",
    log_x=True,
    title="Selling Price Distribution — Log Scale",
    labels={"price_lakh": "Selling Price (₹ Lakh, Log Scale)"}
)

fig.show()


In [37]:
df["log_selling_price"] = np.log1p(df["selling_price"])

In [38]:
brand_counts = (
    df["brand"]
    .value_counts()
    .reset_index()
)

brand_counts.columns = ["brand", "car_count"]

brand_counts.head(15)

,brand,car_count
0,Maruti,4927
1,Hyundai,2951
2,Honda,1475
3,Mahindra,994
4,Toyota,782
5,Ford,774
6,Volkswagen,611
7,Renault,526
8,BMW,436
9,Tata,421


In [39]:
fig = px.bar(
    brand_counts.head(15),
    x="car_count",
    y="brand",
    orientation="h",
    title="Top 15 Brands by Number of Used Cars",
    labels={
        "car_count": "Number of Cars",
        "brand": "Brand"
    }
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [40]:
brand_price = (
    df.groupby("brand")
    .agg(
        car_count=("car_name", "count"),
        avg_price_lakh=("price_lakh", "mean"),
        median_price_lakh=("price_lakh", "median")
    )
    .reset_index()
)

In [41]:
brand_price_filtered = brand_price[
    brand_price["car_count"] >= 50
].copy()

In [42]:
brand_price_filtered = brand_price_filtered.sort_values(
    "median_price_lakh",
    ascending=False
)

brand_price_filtered.head(15)

,brand,car_count,avg_price_lakh,median_price_lakh
14,Land Rover,50,38.468800,34.875
11,Jaguar,58,26.472241,25.000
1,BMW,436,26.866743,22.875
21,Mercedes-Benz,331,24.836465,18.000
0,Audi,190,19.707526,17.000
29,Toyota,782,13.753657,12.775
17,Mahindra,994,7.902364,7.500
7,Honda,1475,6.175153,5.900
27,Skoda,331,7.868943,5.900
6,Ford,774,6.342235,5.660


In [43]:
fig = px.bar(
    brand_price_filtered.head(15),
    x="median_price_lakh",
    y="brand",
    orientation="h",
    title="Top Brands by Median Used Car Selling Price",
    labels={
        "median_price_lakh": "Median Selling Price (₹ Lakh)",
        "brand": "Brand"
    },
    hover_data=["car_count", "avg_price_lakh"]
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [44]:
fig = px.scatter(
    df,
    x="vehicle_age",
    y="price_lakh",
    opacity=0.35,
    title="Vehicle Age vs Selling Price",
    labels={
        "vehicle_age": "Vehicle Age (Years)",
        "price_lakh": "Selling Price (₹ Lakh)"
    },
    hover_data=["brand", "model", "km_driven"]
)

fig.show()

In [45]:
fig = px.scatter(
    df[df["price_lakh"] <= price_99],
    x="vehicle_age",
    y="price_lakh",
    opacity=0.4,
    title="Vehicle Age vs Selling Price — Main Market",
    labels={
        "vehicle_age": "Vehicle Age (Years)",
        "price_lakh": "Selling Price (₹ Lakh)"
    },
    hover_data=["brand", "model", "km_driven"]
)

fig.show()

In [46]:
age_price = (
    df.groupby("vehicle_age")
    .agg(
        car_count=("car_name", "count"),
        avg_price_lakh=("price_lakh", "mean"),
        median_price_lakh=("price_lakh", "median")
    )
    .reset_index()
)

age_price.head(10)

,vehicle_age,car_count,avg_price_lakh,median_price_lakh
0,0,4,15.750000,5.250
1,1,221,10.969774,7.500
2,2,1134,11.430097,8.180
3,3,1896,9.773270,7.480
4,4,2216,9.494959,6.955
5,5,2089,8.203863,6.000
6,6,1900,7.282489,5.500
7,7,1426,7.003184,5.000
8,8,1270,6.274543,4.500
9,9,1012,5.924150,4.000


In [47]:
fig = px.line(
    age_price,
    x="vehicle_age",
    y="median_price_lakh",
    markers=True,
    title="Median Used Car Price by Vehicle Age",
    labels={
        "vehicle_age": "Vehicle Age (Years)",
        "median_price_lakh": "Median Selling Price (₹ Lakh)"
    }
)

fig.show()

In [48]:
age_price.sort_values("vehicle_age")

,vehicle_age,car_count,avg_price_lakh,median_price_lakh
0,0,4,15.750000,5.250
1,1,221,10.969774,7.500
2,2,1134,11.430097,8.180
3,3,1896,9.773270,7.480
4,4,2216,9.494959,6.955
5,5,2089,8.203863,6.000
6,6,1900,7.282489,5.500
7,7,1426,7.003184,5.000
8,8,1270,6.274543,4.500
9,9,1012,5.924150,4.000


In [49]:
age_price_reliable = age_price[
    age_price["car_count"] >= 30
].copy()

In [50]:
fig = px.line(
    age_price_reliable,
    x="vehicle_age",
    y="median_price_lakh",
    markers=True,
    title="Median Used Car Price by Vehicle Age (Minimum 30 Cars per Age)",
    labels={
        "vehicle_age": "Vehicle Age (Years)",
        "median_price_lakh": "Median Selling Price (₹ Lakh)"
    },
    hover_data=["car_count"]
)

fig.show()

In [51]:
age_price_reliable = age_price_reliable.sort_values("vehicle_age")

age_price_reliable["price_change_lakh"] = (
    age_price_reliable["median_price_lakh"].diff()
)

age_price_reliable["depreciation_pct"] = (
    age_price_reliable["median_price_lakh"].pct_change() * -100
)

age_price_reliable[
    [
        "vehicle_age",
        "car_count",
        "median_price_lakh",
        "price_change_lakh",
        "depreciation_pct"
    ]
]

,vehicle_age,car_count,median_price_lakh,price_change_lakh,depreciation_pct
1,1,221,7.500,NaN,NaN
2,2,1134,8.180,0.680,-9.066667
3,3,1896,7.480,-0.700,8.557457
4,4,2216,6.955,-0.525,7.018717
5,5,2089,6.000,-0.955,13.731129
6,6,1900,5.500,-0.500,8.333333
7,7,1426,5.000,-0.500,9.090909
8,8,1270,4.500,-0.500,10.000000
9,9,1012,4.000,-0.500,11.111111
10,10,703,3.360,-0.640,16.000000


In [52]:
fig = px.bar(
    age_price_reliable.dropna(subset=["depreciation_pct"]),
    x="vehicle_age",
    y="depreciation_pct",
    title="Estimated Year-on-Year Market Depreciation by Vehicle Age",
    labels={
        "vehicle_age": "Vehicle Age (Years)",
        "depreciation_pct": "Change in Median Price (%)"
    }
)

fig.show()

In [53]:
brand_counts = df["brand"].value_counts()

major_brands = brand_counts[brand_counts >= 300].index

In [54]:
brand_age_price = (
    df[df["brand"].isin(major_brands)]
    .groupby(["brand", "vehicle_age"])
    .agg(
        car_count=("car_name", "count"),
        median_price_lakh=("price_lakh", "median")
    )
    .reset_index()
)

In [55]:
brand_age_price = brand_age_price[
    brand_age_price["car_count"] >= 20
].copy()

In [56]:
fig = px.line(
    brand_age_price,
    x="vehicle_age",
    y="median_price_lakh",
    color="brand",
    markers=True,
    title="Used Car Price Depreciation by Brand",
    labels={
        "vehicle_age": "Vehicle Age (Years)",
        "median_price_lakh": "Median Selling Price (₹ Lakh)",
        "brand": "Brand"
    },
    hover_data=["car_count"]
)

fig.show()

In [57]:
top_brands = (
    df["brand"]
    .value_counts()
    .head(6)
    .index
)

brand_age_top = brand_age_price[
    brand_age_price["brand"].isin(top_brands)
]

In [58]:
fig = px.line(
    brand_age_top,
    x="vehicle_age",
    y="median_price_lakh",
    color="brand",
    markers=True,
    title="Depreciation Curves — Top 6 Used Car Brands",
    labels={
        "vehicle_age": "Vehicle Age (Years)",
        "median_price_lakh": "Median Selling Price (₹ Lakh)",
        "brand": "Brand"
    },
    hover_data=["car_count"]
)

fig.show()

In [59]:
retention_data = brand_age_top[
    (brand_age_top["vehicle_age"] >= 1) &
    (brand_age_top["vehicle_age"] <= 12)
].copy()

In [60]:
baseline_price = (
    retention_data[retention_data["vehicle_age"] == 1]
    [["brand", "median_price_lakh"]]
    .rename(columns={
        "median_price_lakh": "baseline_price_lakh"
    })
)

baseline_price

,brand,baseline_price_lakh
50,Hyundai,7.25
85,Maruti,6.25


In [61]:
retention_data = retention_data.merge(
    baseline_price,
    on="brand",
    how="inner"
)

In [62]:
retention_data["value_retention_pct"] = (
    retention_data["median_price_lakh"] /
    retention_data["baseline_price_lakh"]
) * 100

In [63]:
retention_data[
    [
        "brand",
        "vehicle_age",
        "median_price_lakh",
        "baseline_price_lakh",
        "value_retention_pct"
    ]
].head(20)

,brand,vehicle_age,median_price_lakh,baseline_price_lakh,value_retention_pct
0,Hyundai,1,7.250,7.25,100.000000
1,Hyundai,2,7.800,7.25,107.586207
2,Hyundai,3,7.500,7.25,103.448276
3,Hyundai,4,6.500,7.25,89.655172
4,Hyundai,5,5.970,7.25,82.344828
5,Hyundai,6,5.250,7.25,72.413793
6,Hyundai,7,4.500,7.25,62.068966
7,Hyundai,8,4.165,7.25,57.448276
8,Hyundai,9,3.725,7.25,51.379310
9,Hyundai,10,2.850,7.25,39.310345


In [64]:
fig = px.line(
    retention_data,
    x="vehicle_age",
    y="value_retention_pct",
    color="brand",
    markers=True,
    title="Relative Used-Car Value Retention by Brand",
    labels={
        "vehicle_age": "Vehicle Age (Years)",
        "value_retention_pct": "Value Retained (%)",
        "brand": "Brand"
    }
)

fig.update_layout(
    yaxis_title="Value Retained vs 1-Year-Old Median Price (%)"
)

fig.show()

In [65]:
retention_5yr = (
    retention_data[retention_data["vehicle_age"] == 5]
    [["brand", "value_retention_pct", "median_price_lakh"]]
    .sort_values("value_retention_pct", ascending=False)
)

retention_5yr

,brand,value_retention_pct,median_price_lakh
16,Maruti,84.800000,5.30
4,Hyundai,82.344828,5.97


In [66]:
fig = px.bar(
    retention_5yr,
    x="value_retention_pct",
    y="brand",
    orientation="h",
    title="5-Year Relative Value Retention by Brand",
    labels={
        "value_retention_pct": "Value Retained (%)",
        "brand": "Brand"
    },
    text_auto=".1f"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [67]:
brand_age_price = (
    df.groupby(["brand", "vehicle_age"])
    .agg(
        car_count=("car_name", "count"),
        median_price_lakh=("price_lakh", "median")
    )
    .reset_index()
)

brand_age_price = brand_age_price[
    brand_age_price["car_count"] >= 10
].copy()

In [68]:
eligible_brands = (
    df["brand"]
    .value_counts()
)

eligible_brands = eligible_brands[
    eligible_brands >= 100
].index

In [69]:
retention_data = brand_age_price[
    (brand_age_price["brand"].isin(eligible_brands)) &
    (brand_age_price["vehicle_age"].between(1, 12))
].copy()

In [70]:
baseline_price = (
    retention_data[
        retention_data["vehicle_age"] == 1
    ][["brand", "median_price_lakh"]]
    .rename(
        columns={"median_price_lakh": "baseline_price_lakh"}
    )
)

In [71]:
retention_data = retention_data.merge(
    baseline_price,
    on="brand",
    how="inner"
)

retention_data["value_retention_pct"] = (
    retention_data["median_price_lakh"] /
    retention_data["baseline_price_lakh"]
) * 100

In [72]:
retention_5yr = (
    retention_data[
        retention_data["vehicle_age"] == 5
    ]
    [["brand", "car_count", "value_retention_pct", "median_price_lakh"]]
    .sort_values(
        "value_retention_pct",
        ascending=False
    )
)

retention_5yr

,brand,car_count,value_retention_pct,median_price_lakh
28,Mahindra,136,101.500000,8.12
40,Maruti,715,84.800000,5.30
16,Hyundai,388,82.344828,5.97
52,Renault,119,72.826087,3.35
4,Honda,219,67.532468,6.50


In [73]:
fig = px.bar(
    retention_5yr,
    x="value_retention_pct",
    y="brand",
    orientation="h",
    title="5-Year Relative Value Retention by Brand",
    labels={
        "value_retention_pct": "Value Retained (%)",
        "brand": "Brand"
    },
    text_auto=".1f",
    hover_data=["car_count", "median_price_lakh"]
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [74]:
fig = px.scatter(
    df,
    x="km_driven",
    y="price_lakh",
    opacity=0.35,
    title="Kilometres Driven vs Selling Price",
    labels={
        "km_driven": "Kilometres Driven",
        "price_lakh": "Selling Price (₹ Lakh)"
    },
    hover_data=["brand", "model", "vehicle_age"]
)

fig.show()

In [75]:
fig = px.scatter(
    df[df["price_lakh"] <= price_99],
    x="km_driven",
    y="price_lakh",
    opacity=0.4,
    title="Kilometres Driven vs Selling Price — Main Market",
    labels={
        "km_driven": "Kilometres Driven",
        "price_lakh": "Selling Price (₹ Lakh)"
    },
    hover_data=["brand", "model", "vehicle_age"]
)

fig.show()

In [76]:
km_price = (
    df.groupby("km_band", observed=True)
    .agg(
        car_count=("car_name", "count"),
        median_price_lakh=("price_lakh", "median"),
        avg_price_lakh=("price_lakh", "mean")
    )
    .reset_index()
)

km_price

,km_band,car_count,median_price_lakh,avg_price_lakh
0,0-25k,3170,6.50,9.932366
1,25k-50k,4680,5.75,8.018468
2,50k-75k,4114,5.25,6.770416
3,75k-100k,2009,5.00,6.437625
4,100k-150k,989,5.20,6.574247
5,150k+,252,5.05,6.377738


In [77]:
fig = px.bar(
    km_price,
    x="km_band",
    y="median_price_lakh",
    title="Median Used Car Price by Kilometres Driven",
    labels={
        "km_band": "Kilometres Driven",
        "median_price_lakh": "Median Selling Price (₹ Lakh)"
    },
    text_auto=".2f",
    hover_data=["car_count"]
)

fig.show()

In [78]:
print(
    "KM Driven vs Price:",
    round(df["km_driven"].corr(df["selling_price"]), 3)
)

print(
    "Vehicle Age vs Price:",
    round(df["vehicle_age"].corr(df["selling_price"]), 3)
)

print(
    "KM Driven vs Vehicle Age:",
    round(df["km_driven"].corr(df["vehicle_age"]), 3)
)

KM Driven vs Price: -0.121
Vehicle Age vs Price: -0.241
KM Driven vs Vehicle Age: 0.49


In [79]:
fuel_analysis = (
    df.groupby("fuel_type")
    .agg(
        car_count=("car_name", "count"),
        median_price_lakh=("price_lakh", "median"),
        avg_price_lakh=("price_lakh", "mean"),
        median_age=("vehicle_age", "median"),
        median_km=("km_driven", "median")
    )
    .reset_index()
    .sort_values("median_price_lakh", ascending=False)
)

fuel_analysis

,fuel_type,car_count,median_price_lakh,avg_price_lakh,median_age,median_km
2,Electric,4,18.575,18.535000,6.0,62000.0
1,Diesel,7320,7.000,10.012978,6.0,62000.0
4,Petrol,7547,4.600,5.724001,5.0,38953.0
0,CNG,299,3.700,4.176890,5.0,53000.0
3,LPG,44,1.825,2.062727,11.0,72260.5


In [80]:
fig = px.bar(
    fuel_analysis,
    x="fuel_type",
    y="median_price_lakh",
    title="Median Used Car Price by Fuel Type",
    labels={
        "fuel_type": "Fuel Type",
        "median_price_lakh": "Median Selling Price (₹ Lakh)"
    },
    text_auto=".2f",
    hover_data=["car_count", "median_age", "median_km"]
)

fig.show()

In [81]:
transmission_analysis = (
    df.groupby("transmission_type")
    .agg(
        car_count=("car_name", "count"),
        median_price_lakh=("price_lakh", "median"),
        avg_price_lakh=("price_lakh", "mean")
    )
    .reset_index()
)

transmission_analysis

,transmission_type,car_count,median_price_lakh,avg_price_lakh
0,Automatic,3147,10.5,15.777432
1,Manual,12067,5.0,5.656439


In [82]:
fig = px.bar(
    transmission_analysis,
    x="transmission_type",
    y="median_price_lakh",
    title="Median Used Car Price by Transmission",
    labels={
        "transmission_type": "Transmission",
        "median_price_lakh": "Median Selling Price (₹ Lakh)"
    },
    text_auto=".2f",
    hover_data=["car_count"]
)

fig.show()

In [83]:
manual_median = df.loc[
    df["transmission_type"] == "Manual",
    "price_lakh"
].median()

automatic_median = df.loc[
    df["transmission_type"] == "Automatic",
    "price_lakh"
].median()

automatic_premium_pct = (
    (automatic_median - manual_median) /
    manual_median
) * 100

print("Manual Median Price:", round(manual_median, 2), "Lakh")
print("Automatic Median Price:", round(automatic_median, 2), "Lakh")
print("Automatic Price Premium:", round(automatic_premium_pct, 2), "%")

Manual Median Price: 5.0 Lakh
Automatic Median Price: 10.5 Lakh
Automatic Price Premium: 110.0 %


In [84]:
seller_analysis = (
    df.groupby("seller_type")
    .agg(
        car_count=("car_name", "count"),
        median_price_lakh=("price_lakh", "median"),
        avg_price_lakh=("price_lakh", "mean")
    )
    .reset_index()
    .sort_values("median_price_lakh", ascending=False)
)

seller_analysis

,seller_type,car_count,median_price_lakh,avg_price_lakh
0,Dealer,9441,5.9,8.705470
2,Trustmark Dealer,173,5.4,5.719595
1,Individual,5600,5.1,6.201785


In [85]:
fig = px.bar(
    seller_analysis,
    x="seller_type",
    y="median_price_lakh",
    title="Median Used Car Price by Seller Type",
    labels={
        "seller_type": "Seller Type",
        "median_price_lakh": "Median Selling Price (₹ Lakh)"
    },
    text_auto=".2f",
    hover_data=["car_count"]
)

fig.show()

In [86]:
performance_corr = df[
    [
        "engine",
        "max_power",
        "mileage",
        "selling_price"
    ]
].corr()

performance_corr

,engine,max_power,mileage,selling_price
engine,1.000000,0.808682,-0.631490,0.586754
max_power,0.808682,1.000000,-0.532917,0.751043
mileage,-0.631490,-0.532917,1.000000,-0.305643
selling_price,0.586754,0.751043,-0.305643,1.000000


In [87]:
print(
    "Engine vs Price:",
    round(df["engine"].corr(df["selling_price"]), 3)
)

print(
    "Max Power vs Price:",
    round(df["max_power"].corr(df["selling_price"]), 3)
)

print(
    "Mileage vs Price:",
    round(df["mileage"].corr(df["selling_price"]), 3)
)

Engine vs Price: 0.587
Max Power vs Price: 0.751
Mileage vs Price: -0.306


In [88]:
fig = px.scatter(
    df[df["price_lakh"] <= price_99],
    x="engine",
    y="price_lakh",
    opacity=0.4,
    title="Engine Size vs Selling Price",
    labels={
        "engine": "Engine Capacity (cc)",
        "price_lakh": "Selling Price (₹ Lakh)"
    },
    hover_data=["brand", "model", "vehicle_age", "max_power"]
)

fig.show()

In [89]:
fig = px.scatter(
    df[df["price_lakh"] <= price_99],
    x="max_power",
    y="price_lakh",
    opacity=0.4,
    title="Maximum Power vs Selling Price",
    labels={
        "max_power": "Maximum Power (bhp)",
        "price_lakh": "Selling Price (₹ Lakh)"
    },
    hover_data=["brand", "model", "engine", "vehicle_age"]
)

fig.show()

In [90]:
fig = px.scatter(
    df[df["price_lakh"] <= price_99],
    x="mileage",
    y="price_lakh",
    opacity=0.4,
    title="Mileage vs Selling Price",
    labels={
        "mileage": "Mileage",
        "price_lakh": "Selling Price (₹ Lakh)"
    },
    hover_data=["brand", "model", "fuel_type"]
)

fig.show()

In [91]:
numeric_cols = [
    "vehicle_age",
    "km_driven",
    "mileage",
    "engine",
    "max_power",
    "seats",
    "selling_price"
]

corr_matrix = df[numeric_cols].corr()

In [92]:
fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Heatmap of Numerical Features",
    labels=dict(
        color="Correlation"
    )
)

fig.show()

In [93]:
features = [
    "vehicle_age",
    "km_driven",
    "mileage",
    "engine",
    "max_power",
    "seats",
    "brand",
    "seller_type",
    "fuel_type",
    "transmission_type"
]

target = "selling_price"

In [94]:
X = df[features].copy()
y = df[target].copy()

In [95]:
numeric_features = [
    "vehicle_age",
    "km_driven",
    "mileage",
    "engine",
    "max_power",
    "seats"
]

categorical_features = [
    "brand",
    "seller_type",
    "fuel_type",
    "transmission_type"
]

In [97]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [98]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 12171
Testing rows: 3043


In [99]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

In [100]:
linear_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['vehicle_age','km_driven','mileage',...,'seller_type','fuel_type', 'transmission_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. T

In [101]:
y_pred_linear = linear_model.predict(X_test)

C:\Users\roder\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [102]:
linear_r2 = r2_score(y_test, y_pred_linear)
linear_mae = mean_absolute_error(y_test, y_pred_linear)
linear_rmse = np.sqrt(mean_squared_error(y_test, y_pred_linear))

print("Linear Regression R²:", round(linear_r2, 4))
print("Linear Regression MAE:", round(linear_mae, 2))
print("Linear Regression RMSE:", round(linear_rmse, 2))

Linear Regression R²: 0.5277
Linear Regression MAE: 218762.54
Linear Regression RMSE: 753417.31


In [103]:
print("MAE:", round(linear_mae / 100000, 2), "Lakh")
print("RMSE:", round(linear_rmse / 100000, 2), "Lakh")

MAE: 2.19 Lakh
RMSE: 7.53 Lakh


In [104]:
linear_results = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": y_pred_linear
})

linear_results["Actual Price (Lakh)"] = linear_results["Actual Price"] / 100000
linear_results["Predicted Price (Lakh)"] = linear_results["Predicted Price"] / 100000

In [105]:
fig = px.scatter(
    linear_results,
    x="Actual Price (Lakh)",
    y="Predicted Price (Lakh)",
    opacity=0.45,
    title="Linear Regression: Actual vs Predicted Selling Price",
    labels={
        "Actual Price (Lakh)": "Actual Price (₹ Lakh)",
        "Predicted Price (Lakh)": "Predicted Price (₹ Lakh)"
    }
)

min_price = min(
    linear_results["Actual Price (Lakh)"].min(),
    linear_results["Predicted Price (Lakh)"].min()
)

max_price = max(
    linear_results["Actual Price (Lakh)"].max(),
    linear_results["Predicted Price (Lakh)"].max()
)

fig.add_shape(
    type="line",
    x0=min_price,
    y0=min_price,
    x1=max_price,
    y1=max_price,
    line=dict(dash="dash")
)

fig.show()

In [106]:
y_log = np.log1p(df["selling_price"])

In [107]:
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X,
    y_log,
    test_size=0.20,
    random_state=42
)

In [108]:
log_linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

log_linear_model.fit(X_train_log, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['vehicle_age','km_driven','mileage',...,'seller_type','fuel_type', 'transmission_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. T

In [109]:
y_pred_log = log_linear_model.predict(X_test_log)

C:\Users\roder\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [110]:
y_pred_log_price = np.expm1(y_pred_log)
y_test_log_price = np.expm1(y_test_log)

In [111]:
log_r2 = r2_score(
    y_test_log_price,
    y_pred_log_price
)

log_mae = mean_absolute_error(
    y_test_log_price,
    y_pred_log_price
)

log_rmse = np.sqrt(
    mean_squared_error(
        y_test_log_price,
        y_pred_log_price
    )
)

print("Log Linear Regression R²:", round(log_r2, 4))
print("MAE:", round(log_mae / 100000, 2), "Lakh")
print("RMSE:", round(log_rmse / 100000, 2), "Lakh")

Log Linear Regression R²: 0.7083
MAE: 1.57 Lakh
RMSE: 5.92 Lakh


In [112]:
log_scale_r2 = r2_score(
    y_test_log,
    y_pred_log
)

print(
    "Log-scale R²:",
    round(log_scale_r2, 4)
)

Log-scale R²: 0.8942


In [113]:
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [114]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [115]:
rf_model.fit(X_train_rf, y_train_rf)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['vehicle_age','km_driven','mileage',...,'seller_type','fuel_type', 'transmission_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. T

In [116]:
y_pred_rf = rf_model.predict(X_test_rf)

C:\Users\roder\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [117]:
rf_r2 = r2_score(y_test_rf, y_pred_rf)
rf_mae = mean_absolute_error(y_test_rf, y_pred_rf)
rf_rmse = np.sqrt(
    mean_squared_error(y_test_rf, y_pred_rf)
)

print("Random Forest R²:", round(rf_r2, 4))
print("MAE:", round(rf_mae / 100000, 2), "Lakh")
print("RMSE:", round(rf_rmse / 100000, 2), "Lakh")

Random Forest R²: 0.767
MAE: 1.16 Lakh
RMSE: 5.29 Lakh


In [118]:
rf_results = pd.DataFrame({
    "Actual Price (Lakh)": y_test_rf / 100000,
    "Predicted Price (Lakh)": y_pred_rf / 100000
})

fig = px.scatter(
    rf_results,
    x="Actual Price (Lakh)",
    y="Predicted Price (Lakh)",
    opacity=0.45,
    title="Random Forest: Actual vs Predicted Selling Price"
)

min_price = min(
    rf_results["Actual Price (Lakh)"].min(),
    rf_results["Predicted Price (Lakh)"].min()
)

max_price = max(
    rf_results["Actual Price (Lakh)"].max(),
    rf_results["Predicted Price (Lakh)"].max()
)

fig.add_shape(
    type="line",
    x0=min_price,
    y0=min_price,
    x1=max_price,
    y1=max_price,
    line=dict(dash="dash")
)

fig.show()

In [119]:
gb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            GradientBoostingRegressor(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            )
        )
    ]
)

In [120]:
gb_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['vehicle_age','km_driven','mileage',...,'seller_type','fuel_type', 'transmission_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. T

In [121]:
y_pred_gb = gb_model.predict(X_test)

C:\Users\roder\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [122]:
gb_r2 = r2_score(y_test, y_pred_gb)
gb_mae = mean_absolute_error(y_test, y_pred_gb)
gb_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_gb)
)

print("Gradient Boosting R²:", round(gb_r2, 4))
print("MAE:", round(gb_mae / 100000, 2), "Lakh")
print("RMSE:", round(gb_rmse / 100000, 2), "Lakh")

Gradient Boosting R²: 0.8086
MAE: 1.29 Lakh
RMSE: 4.8 Lakh


In [123]:
model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Log Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "R2": [
        linear_r2,
        log_r2,
        rf_r2,
        gb_r2
    ],
    "MAE_Lakh": [
        linear_mae / 100000,
        log_mae / 100000,
        rf_mae / 100000,
        gb_mae / 100000
    ],
    "RMSE_Lakh": [
        linear_rmse / 100000,
        log_rmse / 100000,
        rf_rmse / 100000,
        gb_rmse / 100000
    ]
})

model_comparison

,Model,R2,MAE_Lakh,RMSE_Lakh
0,Linear Regression,0.527717,2.187625,7.534173
1,Log Linear Regression,0.708260,1.566089,5.921508
2,Random Forest,0.767044,1.157533,5.291407
3,Gradient Boosting,0.808591,1.290326,4.796406


In [124]:
fig = px.bar(
    model_comparison.sort_values("R2"),
    x="R2",
    y="Model",
    orientation="h",
    title="Regression Model Performance Comparison",
    text_auto=".3f"
)

fig.show()

In [125]:
feature_names = gb_model.named_steps["preprocessor"].get_feature_names_out()

In [126]:
gb_importance = gb_model.named_steps["regressor"].feature_importances_

In [127]:
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": gb_importance
})

In [128]:
feature_importance["feature"] = (
    feature_importance["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

In [129]:
feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance.head(15)

,feature,importance
0,max_power,0.706033
1,vehicle_age,0.155515
2,km_driven,0.057492
3,engine,0.027544
4,brand_Land Rover,0.009177
5,transmission_type_Manual,0.006853
6,mileage,0.006781
7,brand_Bentley,0.006193
8,brand_Mercedes-Benz,0.005861
9,brand_Rolls-Royce,0.004735


In [130]:
fig = px.bar(
    feature_importance.head(15).sort_values("importance"),
    x="importance",
    y="feature",
    orientation="h",
    title="Top Drivers of Used Car Selling Price",
    labels={
        "importance": "Feature Importance",
        "feature": "Feature"
    },
    text_auto=".3f"
)

fig.show()

In [131]:
feature_importance.head(10)

,feature,importance
0,max_power,0.706033
1,vehicle_age,0.155515
2,km_driven,0.057492
3,engine,0.027544
4,brand_Land Rover,0.009177
5,transmission_type_Manual,0.006853
6,mileage,0.006781
7,brand_Bentley,0.006193
8,brand_Mercedes-Benz,0.005861
9,brand_Rolls-Royce,0.004735


In [132]:
gb_results = X_test.copy()

gb_results["actual_price"] = y_test.values
gb_results["predicted_price"] = y_pred_gb

In [133]:
gb_results["price_gap"] = (
    gb_results["actual_price"] -
    gb_results["predicted_price"]
)

gb_results["price_gap_lakh"] = (
    gb_results["price_gap"] / 100000
)

In [134]:
gb_results["price_gap_pct"] = (
    gb_results["price_gap"] /
    gb_results["predicted_price"]
) * 100

In [135]:
def classify_price_gap(x):
    if x <= -15:
        return "Potentially Underpriced"
    elif x >= 15:
        return "Potentially Overpriced"
    else:
        return "Fairly Priced"

gb_results["price_position"] = (
    gb_results["price_gap_pct"]
    .apply(classify_price_gap)
)

In [136]:
gb_results["price_position"].value_counts()

price_position
Fairly Priced              1776
Potentially Underpriced     673
Potentially Overpriced      594
Name: count, dtype: int64

In [137]:
price_position_counts = (
    gb_results["price_position"]
    .value_counts()
    .reset_index()
)

price_position_counts.columns = [
    "price_position",
    "car_count"
]

fig = px.bar(
    price_position_counts,
    x="price_position",
    y="car_count",
    title="Model-Based Used Car Pricing Position",
    labels={
        "price_position": "Pricing Classification",
        "car_count": "Number of Cars"
    },
    text_auto=True
)

fig.show()

In [138]:
gb_results[
    [
        "brand",
        "vehicle_age",
        "km_driven",
        "actual_price",
        "predicted_price",
        "price_gap_lakh",
        "price_gap_pct"
    ]
].sort_values("price_gap_pct").head(15)

,brand,vehicle_age,km_driven,actual_price,predicted_price,price_gap_lakh,price_gap_pct
7314,Honda,19,110000,50000,258430.026525,-2.084300,-80.652403
11089,Maruti,15,60500,60000,275661.873293,-2.156619,-78.234204
4675,Mahindra,9,15000,153000,536326.799557,-3.833268,-71.472617
4117,Tata,14,74555,105000,282587.717899,-1.775877,-62.843396
4447,Tata,12,270000,120000,319121.890901,-1.991219,-62.396813
1957,Mahindra,18,120000,100000,263344.364537,-1.633444,-62.026907
8782,Hyundai,14,120000,120000,306656.574205,-1.866566,-60.868277
6330,Mahindra,17,120000,145000,368960.555628,-2.239606,-60.700406
5990,Mahindra,18,198565,150000,368960.555628,-2.189606,-59.345248
7479,Hyundai,11,50000,199000,485498.576459,-2.864986,-59.011208


In [139]:
gb_results[
    [
        "brand",
        "vehicle_age",
        "km_driven",
        "actual_price",
        "predicted_price",
        "price_gap_lakh",
        "price_gap_pct"
    ]
].sort_values(
    "price_gap_pct",
    ascending=False
).head(15)

,brand,vehicle_age,km_driven,actual_price,predicted_price,price_gap_lakh,price_gap_pct
13632,Skoda,4,48592,1675000,5.024014e+05,11.725986,233.398752
3039,Skoda,7,59864,1100000,3.508364e+05,7.491636,213.536423
3482,Land Rover,8,75000,7800000,3.101616e+06,46.983843,151.481837
3785,Ferrari,2,3800,39500000,1.720786e+07,222.921424,129.546297
6075,Skoda,11,300000,400000,1.851688e+05,2.148312,116.019145
15061,Mercedes-Benz,11,140000,2200000,1.161026e+06,10.389742,89.487613
1385,Audi,4,49000,2650000,1.470281e+06,11.797191,80.237662
12819,Toyota,7,44000,1550000,8.678491e+05,6.821509,78.602469
12319,Hyundai,3,22886,1785000,1.012061e+06,7.729388,76.372735
6022,Skoda,18,133000,275000,1.570026e+05,1.179974,75.156299


In [140]:
fig = px.scatter(
    gb_results,
    x="predicted_price",
    y="actual_price",
    color="price_position",
    opacity=0.55,
    title="Actual vs Predicted Price with Pricing Classification",
    labels={
        "predicted_price": "Predicted Price (₹)",
        "actual_price": "Actual Selling Price (₹)",
        "price_position": "Pricing Position"
    },
    hover_data=[
        "brand",
        "vehicle_age",
        "km_driven",
        "price_gap_pct"
    ]
)

fig.show()

In [141]:
gb_results["absolute_error"] = abs(
    gb_results["actual_price"] - gb_results["predicted_price"]
)

gb_results["absolute_error_lakh"] = (
    gb_results["absolute_error"] / 100000
)

In [142]:
brand_error = (
    gb_results.groupby("brand")
    .agg(
        car_count=("brand", "count"),
        mae_lakh=("absolute_error_lakh", "mean"),
        median_error_lakh=("absolute_error_lakh", "median"),
        avg_actual_price=("actual_price", "mean")
    )
    .reset_index()
)

In [143]:
brand_error_filtered = brand_error[
    brand_error["car_count"] >= 20
].copy()

In [144]:
brand_error_filtered = brand_error_filtered.sort_values(
    "mae_lakh",
    ascending=False
)

brand_error_filtered

,brand,car_count,mae_lakh,median_error_lakh,avg_actual_price
1,BMW,88,4.512699,3.686262,2.724773e+06
0,Audi,34,4.422139,3.728430,1.843824e+06
18,Mercedes-Benz,72,3.750691,2.411071,2.425250e+06
23,Skoda,79,1.748344,1.175603,8.939620e+05
25,Toyota,158,1.581189,1.294512,1.358259e+06
15,Mahindra,211,1.215904,1.040315,7.825687e+05
24,Tata,66,1.027789,0.714651,6.953030e+05
5,Ford,165,0.979750,0.686798,6.296606e+05
6,Honda,320,0.918930,0.662211,6.237188e+05
7,Hyundai,545,0.891368,0.687585,5.941560e+05


In [145]:
fig = px.bar(
    brand_error_filtered.sort_values("mae_lakh"),
    x="mae_lakh",
    y="brand",
    orientation="h",
    title="Prediction Error by Brand",
    labels={
        "mae_lakh": "Mean Absolute Error (₹ Lakh)",
        "brand": "Brand"
    },
    hover_data=[
        "car_count",
        "median_error_lakh",
        "avg_actual_price"
    ],
    text_auto=".2f"
)

fig.show()

In [146]:
gb_results["absolute_pct_error"] = (
    abs(
        gb_results["actual_price"] -
        gb_results["predicted_price"]
    )
    / gb_results["actual_price"]
) * 100

In [147]:
brand_pct_error = (
    gb_results.groupby("brand")
    .agg(
        car_count=("brand", "count"),
        mape_pct=("absolute_pct_error", "mean")
    )
    .reset_index()
)

brand_pct_error = brand_pct_error[
    brand_pct_error["car_count"] >= 20
].sort_values("mape_pct")

In [148]:
fig = px.bar(
    brand_pct_error,
    x="mape_pct",
    y="brand",
    orientation="h",
    title="Percentage Prediction Error by Brand",
    labels={
        "mape_pct": "Mean Absolute Percentage Error (%)",
        "brand": "Brand"
    },
    text_auto=".1f"
)

fig.show()

In [149]:
model_summary = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Log Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "R2": [
        linear_r2,
        log_r2,
        rf_r2,
        gb_r2
    ],
    "MAE_Lakh": [
        linear_mae / 100000,
        log_mae / 100000,
        rf_mae / 100000,
        gb_mae / 100000
    ],
    "RMSE_Lakh": [
        linear_rmse / 100000,
        log_rmse / 100000,
        rf_rmse / 100000,
        gb_rmse / 100000
    ]
}).sort_values("R2", ascending=False)

model_summary

,Model,R2,MAE_Lakh,RMSE_Lakh
3,Gradient Boosting,0.808591,1.290326,4.796406
2,Random Forest,0.767044,1.157533,5.291407
1,Log Linear Regression,0.708260,1.566089,5.921508
0,Linear Regression,0.527717,2.187625,7.534173


In [150]:
fig = px.bar(
    model_summary.sort_values("MAE_Lakh", ascending=False),
    x="MAE_Lakh",
    y="Model",
    orientation="h",
    title="Model Comparison by Mean Absolute Error",
    labels={
        "MAE_Lakh": "MAE (₹ Lakh)",
        "Model": "Model"
    },
    text_auto=".2f"
)

fig.show()

In [151]:
fig = px.bar(
    model_summary.sort_values("RMSE_Lakh", ascending=False),
    x="RMSE_Lakh",
    y="Model",
    orientation="h",
    title="Model Comparison by RMSE",
    labels={
        "RMSE_Lakh": "RMSE (₹ Lakh)",
        "Model": "Model"
    },
    text_auto=".2f"
)

fig.show()

In [152]:
business_insights = pd.DataFrame({
    "Insight": [
        "Vehicle age has stronger negative association with price than kilometres driven",
        "Maximum power is the strongest numerical price correlate",
        "Engine size has a strong positive relationship with resale price",
        "Automatic vehicles have a substantially higher raw median selling price",
        "Mileage has a negative association with selling price",
        "Gradient Boosting delivers the strongest overall model performance",
        "Model accuracy varies across brands"
    ],
    "Finding": [
        "Age vs Price correlation = -0.241; KM vs Price = -0.121",
        "Max Power vs Price correlation = 0.751",
        "Engine vs Price correlation = 0.587",
        "Automatic median ₹10.5L vs Manual ₹5.0L; raw premium 110%",
        "Mileage vs Price correlation = -0.306",
        "R² = 0.8086; MAE = ₹1.29L; RMSE = ₹4.80L",
        "Best MAPE: Audi, Datsun, Skoda; weakest: Toyota, Maruti, Renault"
    ]
})

business_insights

,Insight,Finding
0,Vehicle age has stronger negative association ...,Age vs Price correlation = -0.241; KM vs Price...
1,Maximum power is the strongest numerical price...,Max Power vs Price correlation = 0.751
2,Engine size has a strong positive relationship...,Engine vs Price correlation = 0.587
3,Automatic vehicles have a substantially higher...,Automatic median ₹10.5L vs Manual ₹5.0L; raw p...
4,Mileage has a negative association with sellin...,Mileage vs Price correlation = -0.306
5,Gradient Boosting delivers the strongest overa...,R² = 0.8086; MAE = ₹1.29L; RMSE = ₹4.80L
6,Model accuracy varies across brands,"Best MAPE: Audi, Datsun, Skoda; weakest: Toyot..."


In [153]:
df.to_csv("cardekho_cleaned.csv", index=False)

In [154]:
gb_results.to_csv("cardekho_model_predictions.csv", index=False)

In [155]:
age_price_reliable.to_csv(
    "cardekho_age_depreciation.csv",
    index=False
)

retention_data.to_csv(
    "cardekho_brand_value_retention.csv",
    index=False
)

In [156]:
brand_price_filtered.to_csv(
    "cardekho_brand_pricing.csv",
    index=False
)

fuel_analysis.to_csv(
    "cardekho_fuel_analysis.csv",
    index=False
)

transmission_analysis.to_csv(
    "cardekho_transmission_analysis.csv",
    index=False
)

seller_analysis.to_csv(
    "cardekho_seller_analysis.csv",
    index=False
)

In [157]:
model_summary.to_csv(
    "cardekho_model_summary.csv",
    index=False
)

feature_importance.to_csv(
    "cardekho_feature_importance.csv",
    index=False
)

brand_error_filtered.to_csv(
    "cardekho_brand_error.csv",
    index=False
)

business_insights.to_csv(
    "cardekho_business_insights.csv",
    index=False
)

In [158]:
import os

[f for f in os.listdir() if f.startswith("cardekho")]

['cardekho_age_depreciation.csv',
 'cardekho_brand_error.csv',
 'cardekho_brand_pricing.csv',
 'cardekho_brand_value_retention.csv',
 'cardekho_business_insights.csv',
 'cardekho_cleaned.csv',
 'cardekho_dataset.csv',
 'cardekho_feature_importance.csv',
 'cardekho_fuel_analysis.csv',
 'cardekho_model_predictions.csv',
 'cardekho_model_summary.csv',
 'cardekho_seller_analysis.csv',
 'cardekho_transmission_analysis.csv']